# Genetic Ancestry Inference Across the Corpus

singlify infers sample-level genetic ancestry from common SNP patterns in
scRNA-seq data. This enables corpus-wide demographic characterization
without requiring genotype data.

**Method**: Reference allele frequencies from 1000 Genomes super-populations
(AFR, AMR, EAS, EUR, SAS) are compared against observed SNP profiles in each sample.

In [1]:
import json, glob
from pathlib import Path
from collections import Counter
import pandas as pd
import numpy as np

# Collect ancestry calls from all processed samples
all_dirs = glob.glob('/mnt/projects/debruinz_project/singlify_pipeline/quant/scrna/GSE*/GSE*/GSM*')
ancestry_samples = [d for d in all_dirs if (Path(d) / 'ancestry_call.json').exists()]
print(f'Found {len(ancestry_samples)} samples with ancestry data')

records = []
for d in ancestry_samples:
    a = json.load(open(Path(d) / 'ancestry_call.json'))
    a['gsm'] = Path(d).name
    records.append(a)

df = pd.DataFrame(records)
print(f'\nAncestry distribution:')
print(df['ancestry'].value_counts().to_string())

Found 605 samples with ancestry data

Ancestry distribution:
ancestry
insufficient_data    479
EUR                   80
EAS                   16
AFR                   14
AMR                   14
SAS                    2


In [2]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# Filter to high-confidence calls
called = df[df['ancestry'] != 'insufficient_data'].copy()
print(f'High-confidence calls: {len(called)} / {len(df)} ({len(called)/len(df):.0%})')
print(f'\nConfidence distribution for called samples:')
print(called['confidence'].describe())

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Pie chart of ancestry
counts = called['ancestry'].value_counts()
colors = {'EUR': '#3b82f6', 'EAS': '#22c55e', 'AFR': '#f59e0b', 'AMR': '#ef4444', 'SAS': '#8b5cf6'}
axes[0].pie(counts.values, labels=counts.index, autopct='%1.1f%%',
            colors=[colors.get(x, '#6b7280') for x in counts.index])
axes[0].set_title(f'Ancestry Distribution (n={len(called)} samples)')

# Confidence histogram
axes[1].hist(called['confidence'], bins=20, color='#3b82f6', edgecolor='white')
axes[1].axvline(0.8, color='red', linestyle='--', label='High confidence threshold')
axes[1].set_xlabel('Confidence Score')
axes[1].set_ylabel('Samples')
axes[1].set_title('Ancestry Call Confidence')
axes[1].legend()

plt.tight_layout()
plt.savefig('ancestry.png', dpi=150, bbox_inches='tight')
plt.show()

High-confidence calls: 126 / 605 (21%)

Confidence distribution for called samples:
count    126.000000
mean       0.799723
std        0.225810
min        0.240808
25%        0.653253
50%        0.900489
75%        0.999749
max        1.000000
Name: confidence, dtype: float64


In [3]:
# Show high-confidence examples from each population
print('\n═══ High-Confidence Examples (conf > 0.8) ═══\n')
high_conf = called[called['confidence'] > 0.8]
for pop in ['EUR', 'EAS', 'AFR', 'AMR', 'SAS']:
    subset = high_conf[high_conf['ancestry'] == pop]
    if len(subset) > 0:
        print(f'{pop}: {len(subset)} samples (e.g., {subset.iloc[0]["gsm"]}, conf={subset.iloc[0]["confidence"]:.2f})')
    else:
        print(f'{pop}: 0 high-confidence samples')

print(f'\n{len(high_conf)} total high-confidence calls ({len(high_conf)/len(df):.0%} of samples with ancestry data)')


═══ High-Confidence Examples (conf > 0.8) ═══

EUR: 56 samples (e.g., GSM7103267, conf=1.00)
EAS: 4 samples (e.g., GSM7840583, conf=0.90)
AFR: 8 samples (e.g., GSM4037632, conf=0.99)
AMR: 6 samples (e.g., GSM7036797, conf=1.00)
SAS: 0 high-confidence samples

74 total high-confidence calls (12% of samples with ancestry data)


## Key Findings

- **Coverage**: Ancestry can be inferred from ~21% of scRNA-seq samples (those with sufficient SNP coverage)
- **Bias**: EUR is over-represented (63%), reflecting known population bias in GEO submissions
- **Confidence**: Most calls have moderate confidence (0.4-1.0); very high confidence (>0.9) indicates clear population signal
- **Applications**: Population-aware batch correction, ancestry-stratified QC thresholds, diversity auditing

This metadata is automatically included when loading with `singlet.load_dir()` — accessible via `adata.uns['ancestry']`.